# MARL leak-free, retrained on GPU

The first leak-free run diverged: the policy is stochastic, its samples saturated the tanh, and the
gradient vanished. The trainer of this repository now uses the mean action, a stronger penalty on the
actions, plain cosine decay and a lower learning rate.

Runtime: choose a GPU (Runtime > Change runtime type > T4 GPU), then Run all.


In [ ]:
# code and data of the repository, without the weights (sparse checkout)
!git clone --quiet --filter=blob:none --no-checkout https://github.com/medhayani/TaylorCouetteML.git /content/tcml
%cd /content/tcml
!git sparse-checkout init --cone --quiet
!git sparse-checkout set train code data leakfree
!git checkout --quiet main
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
!ls data/rl_windows


In [ ]:
import subprocess, sys, time, json
from pathlib import Path

EPOCHS, BATCH, LR, N_SEEDS = 1500, 16, 1e-4, 5
W = 'data/rl_windows/rl_switch_windows_lf_'
OUT = Path('runs/marl_lf2'); OUT.mkdir(parents=True, exist_ok=True)

for i in range(N_SEEDS):
    seed = 42 + i
    t0 = time.time()
    rc = subprocess.call([sys.executable, 'train/train_marl_v2.py',
                          '--windows_train', W + 'train.npz', '--windows_val', W + 'val.npz',
                          '--epochs', str(EPOCHS), '--batch', str(BATCH), '--lr', str(LR),
                          '--n_seeds', '1', '--seed_base', str(seed), '--out_dir', str(OUT)])
    print('seed', seed, 'exit', rc, 'in', round((time.time() - t0) / 60, 1), 'min', flush=True)
    if rc != 0:
        break

for s in sorted(OUT.glob('seed_*')):
    h = json.loads((s / 'history.json').read_text())
    v = [r['val_mae'] for r in h]
    print(s.name, 'epochs', len(h), '| first', round(v[0], 5), '| best', round(min(v), 5),
          'at', v.index(min(v)) + 1, '| last', round(v[-1], 5))


In [ ]:
# evaluation on the 64 held-out elasticities, as for SARL
import sys, json
import numpy as np, pandas as pd, torch
sys.path.insert(0, 'code'); sys.path.insert(0, '.')
from data_pipeline.dataset import HydraWindowsDataset
from models.marl_3sac.marl_model import MARLProSystem
import yaml

S = np.linspace(0, 1, 101); TAPER = 0.20
R6 = lambda e: round(float(e), 6)
cfg = yaml.safe_load(open('code/configs/sizes.yaml'))['marl_pro']

def reconstruct(actions, T):
    a_loc, a_sh, a_geo = actions
    s = torch.linspace(0, 1, T).unsqueeze(0).expand(a_loc.size(0), T)
    dc = a_loc[:, 0:1] * 0.5 + 0.5
    da = a_loc[:, 1:2] * 0.5
    bump = da * torch.exp(-((s - dc) ** 2) / (2 * 0.05 ** 2))
    sh = a_sh[:, 0:1] + a_sh[:, 1:2] * bump
    sl, sr, w, asy = a_geo[:, 0:1], a_geo[:, 1:2], a_geo[:, 2:3], a_geo[:, 3:4]
    geo = (sl * (s - dc).clamp(max=0) + sr * (s - dc).clamp(min=0)
           + w * (s - dc).abs() + asy * (s - 0.5))
    return sh + 0.1 * geo

ds = HydraWindowsDataset('data/rl_windows/rl_switch_windows_lf_test.npz')
obs, sv = torch.from_numpy(ds.obs_seq), torch.from_numpy(ds.static_vec)
corrs, vmae = [], []
for ck in sorted(Path('runs/marl_lf2').glob('seed_*/best.pt')):
    m = MARLProSystem(obs_seq_dim=obs.shape[2], obs_seq_T=obs.shape[1], static_dim=sv.shape[1], cfg=cfg)
    m.load_state_dict(torch.load(ck, map_location='cpu')['state_dict']); m.eval()
    with torch.no_grad():
        h = m.encode(obs, sv)
        acts = [torch.tanh(ag.actor.mean(ag.actor.body(h[:, i, :]))) for i, ag in enumerate(m.agents)]
        corrs.append(reconstruct(acts, obs.shape[1]).numpy())
    v = json.loads((ck.parent / 'history.json').read_text())
    vmae.append(min(r['val_mae'] for r in v if r.get('val_mae') is not None))
w_seed = 1.0 / (np.array(vmae) + 1e-6); w_seed /= w_seed.sum()
corr = np.einsum('i...,i->...', np.stack(corrs), w_seed)
print(len(corrs), 'seeds, weights', np.round(w_seed, 3))

yp, yt = ds.y_pred, ds.y_true
print('window level: base MAE %.5f -> MARL %.5f' % (np.mean(np.abs(yp - yt)), np.mean(np.abs(yp + corr - yt))))

desc = pd.read_csv('data/branch_functional_descriptors_leakfree_all.csv')
med = np.load('data/ensemble_median_targets_lf.npz')
mkey = {(R6(e), int(b)): i for i, (e, b) in enumerate(zip(med['E'], med['branch_local_id']))}
wkey = {(R6(e), int(b)): i for i, (e, b) in enumerate(zip(ds.E, ds.branch_local_id))}
raw = pd.read_csv('data/combined_data.csv'); raw.columns = ['Ta', 'k', 'E']
split = pd.read_csv('data/split_by_E.csv'); test_E = set(np.round(split.loc[split.split == 'test', 'E'].to_numpy(float), 6))
rows = []
for E, g in desc.groupby('E'):
    E = float(E)
    if R6(E) not in test_E: continue
    kp, Tb, Tc = [], [], []
    for _, r in g.iterrows():
        key = (R6(E), int(r.branch_local_id)); base = med['ta_norm_median'][mkey[key]].copy(); ref = base.copy()
        if key in wkey:
            i = wkey[key]; c0 = float(ds.center_pred[i]); hw = float(ds.window_half_width[i])
            cf = np.interp(S, np.clip(c0 + hw * ds.local_grid[i], 0, 1), corr[i], left=0.0, right=0.0)
            d = np.abs(S - c0) / max(hw, 1e-9)
            wt = np.where(d <= 1 - TAPER, 1.0, np.where(d >= 1, 0.0, 0.5 * (1 + np.cos(np.pi * (d - (1 - TAPER)) / TAPER))))
            ref = np.clip(base + wt * cf, 0.0, 1.0)
        k_b = r.k_left + S * (r.k_right - r.k_left); amp = r.Ta_max - r.Ta_min
        kp.append(k_b); Tb.append(r.Ta_min + base * amp); Tc.append(r.Ta_min + ref * amp)
    kp, Tb, Tc = np.concatenate(kp), np.concatenate(Tb), np.concatenate(Tc)
    o = np.argsort(kp); kp, Tb, Tc = kp[o], Tb[o], Tc[o]
    gt = raw[np.isclose(raw.E, E)].sort_values('k'); kt, Tt = gt.k.values, gt.Ta.values; it = np.argmin(Tt)
    cm = lambda T: (lambda Ti: 100 * np.mean(np.abs(Ti[~np.isnan(Ti)] / Tt[~np.isnan(Ti)] - 1)))(np.interp(kt, kp, T, left=np.nan, right=np.nan))
    rows.append(dict(E=E, Ta_c_true=Tt[it], k_c_true=kt[it],
                     Ta_c_base=Tb[np.argmin(Tb)], k_c_base=kp[np.argmin(Tb)], curve_base=cm(Tb),
                     Ta_c_marl=Tc[np.argmin(Tc)], k_c_marl=kp[np.argmin(Tc)], curve_marl=cm(Tc)))
d = pd.DataFrame(rows)
mape = lambda x, t: float(100 * np.mean(np.abs(x / t - 1)))
res = dict(n_test_E=len(d), seeds=len(corrs),
           base=dict(Ta_c=mape(d.Ta_c_base, d.Ta_c_true), k_c=float((d.k_c_base - d.k_c_true).abs().mean()), curve=float(d.curve_base.mean())),
           marl=dict(Ta_c=mape(d.Ta_c_marl, d.Ta_c_true), k_c=float((d.k_c_marl - d.k_c_true).abs().mean()), curve=float(d.curve_marl.mean())))
print(json.dumps(res, indent=1))
d.to_csv('runs/marl_lf2/per_E_marl.csv', index=False)
json.dump(res, open('runs/marl_lf2/metrics.json', 'w'), indent=1)


In [ ]:
# the weights and the metrics, as one archive
!cd runs && zip -qr /content/marl_lf2.zip marl_lf2
from google.colab import files
files.download('/content/marl_lf2.zip')
